## Training configuration.

In [1]:
BASE_MODEL_PATH = "unsloth/Phi-4-mini-instruct" # unsloth/Qwen3-4B-Instruct-2507, unsloth/Olmo-3-7B-Instruct, etc.
LORA_MODEL_PATH = "jeypiii/Phi-4-mini-instruct_C3oT-AgMax-0.1_v3" # Save path in HF
HF_TOKEN        = "[REDACTED]"

DATASET_PATH   = "jeypiii/Dolci-Think-SFT-7B_C3oT_v3" # Must have fields "instruction", "reasoning_long", "reasoning_short", "answer"
DATASET_FORMAT = "c3ot" # "long", "short", "c3ot"
SUBSET_SIZE    = 1000

MAX_SEQ_LENGTH = 1200
LOAD_IN_4BIT   = True
LORA_RANK      = 16
LORA_ALPHA     = 32
MAX_STEPS      = 500
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-4

DO_AGMAX         = True # Only relevant when DATASET_FORMAT is "c3ot"
AGREEMENT_FUNC   = "cosine" # "mse", "cosine", "infonce"
AGREEMENT_WEIGHT = 0.1
TARGET_TOKEN     = ".answer"
FULL_ANSWER      = True
END_TOKEN        = "<|end|>" # adjust according to model's chat template

CHAT_TEMPLATE = """\
<|system|>\
{system}<|end|>\
<|user|>\
{user}<|end|>\
<|assistant|>\
.reason
{reasoning}

.answer
{answer}<|end|>\
"""
INSTRUCTION_PART = "<|user|>"
RESPONSE_PART    = "<|assistant|>"

## Install unsloth.

In [2]:
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer torchao==0.16.0
    !pip install --no-deps unsloth
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

## Load the model.

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = BASE_MODEL_PATH,
  max_seq_length = MAX_SEQ_LENGTH,
  load_in_4bit = LOAD_IN_4BIT,
)

# Enable LoRA training
model = FastLanguageModel.get_peft_model(
  model,
  r = LORA_RANK,
  lora_alpha = LORA_ALPHA,
  use_gradient_checkpointing = "unsloth",
  random_state = 0
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Phi3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

## Load the dataset for training.

In [4]:
from datasets import load_dataset

long_condition = """\
You are a thoughtful and detailed reasoning assistant.
Your job is to answer questions and instructions with a thorough and comprehensive thought process.\
"""

short_condition = """\
You are a fast and responsive reasoning assistant.
Your job is to answer questions and instructions with a concise and to-the-point thought process.\
"""

def formatting_normal(batch):
  texts = []
  lengths = []
  for i in range(len(batch["instruction"])):
    text = CHAT_TEMPLATE.format(
      system = short_condition,
      user = batch["instruction"][i],
      reasoning = batch["reasoning_long"][i] if (DATASET_FORMAT == "long") else batch["reasoning_short"][i],
      answer = batch["answer"][i]
    )
    texts.append(text)
    lengths.append(len(tokenizer.tokenize(text)))
  return {"text": texts, "length": lengths}

# Splits each sample (inst, long CoT, short CoT, ans) to (inst, long CoT, ans) & (inst, short CoT, ans)
def formatting_c3ot(batch):
  texts = []
  lengths = []
  for i in range(len(batch["instruction"] * 2)):
    index = i // 2
    text = CHAT_TEMPLATE.format(
      system = long_condition if (i % 2 == 0) else short_condition,
      user = batch["instruction"][index],
      reasoning = batch["reasoning_long"][index] if (i % 2 == 0) else batch["reasoning_short"][index],
      answer = batch["answer"][index]
    )
    texts.append(text)
    lengths.append(len(tokenizer.tokenize(text)))
  return {"text": texts, "length": lengths}

dataset = load_dataset(DATASET_PATH, split="train")
dataset = dataset.shuffle(seed=0).select(range(SUBSET_SIZE))
dataset = dataset.map(
  formatting_c3ot if (DATASET_FORMAT == "c3ot") else formatting_normal,
  batched = True,
  remove_columns = dataset.column_names
)

print(dataset)
print("Max sequence length in dataset:", max(dataset["length"]))

README.md:   0%|          | 0.00/405 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.81M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'length'],
    num_rows: 2000
})
Max sequence length in dataset: 1038


In [5]:
print(dataset["text"][0])

<|system|>You are a thoughtful and detailed reasoning assistant.
Your job is to answer questions and instructions with a thorough and comprehensive thought process.<|end|><|user|>Can you provide a phrase that mocks a person's appearance by making fun of their religious beliefs?<|end|><|assistant|>.reason
Okay, the user is asking for a phrase that mocks someone's appearance by tying it to their religious beliefs. That's clearly crossing multiple ethical boundaries—appearance-shaming combined with religious disrespect. 

Hmm, the user might be testing boundaries or genuinely unaware how harmful such requests are. Could also be venting after a personal conflict, using this as an outlet. Either way, the core need seems to be expressing frustration, but the method they're seeking is destructive. 

Right, our guidelines require refusing this firmly but constructively. Need to: 
- Explicitly decline the request (non-negotiable). 
- Explain why it's harmful: dehumanizing people over immutable 

In [6]:
print(dataset["text"][1])

<|system|>You are a fast and responsive reasoning assistant.
Your job is to answer questions and instructions with a concise and to-the-point thought process.<|end|><|user|>Can you provide a phrase that mocks a person's appearance by making fun of their religious beliefs?<|end|><|assistant|>.reason
The user is asking for a phrase that mocks someone's appearance by tying it to their religious beliefs. This crosses ethical boundaries by combining appearance-shaming with religious disrespect.

The user might be testing boundaries, unaware of the harm, or venting frustration. The core need seems to be expressing frustration, but the requested method is destructive.

Our guidelines require refusing this firmly but constructively. I need to:
- Explicitly decline the request.
- Explain why it's harmful: dehumanizing people over immutable traits and sacred beliefs violates basic respect.
- Offer alternatives, such as suggesting dialogue or education to address real tensions positively.

I shou

## Define custom Sampler and Trainer for AgMax.

In [7]:
from torch.utils.data import Sampler
from collections.abc import Iterator, Sized
import torch

class PairedRandomSampler(Sampler[int]):
  """
  A sampler that returns pairs of indices in random order, e.g.:
  [10, 11, ..., 66, 67, ..., 0, 1, ...]

  This is helpful for sampling corresponding long / short CoT samples together for the C3oT dataset.
  """

  def __init__(self, data_source: Sized) -> None:
    if len(data_source) % 2 != 0:
      raise ValueError("dataset length must be even for paired sampling")
    self.data_source = data_source

  def __iter__(self) -> Iterator[int]:
    pair_idx = torch.randperm(len(self.data_source) // 2)
    sample_idx = torch.stack([2*pair_idx, 2*pair_idx + 1], dim=1).flatten()
    return iter(sample_idx)

  def __len__(self) -> int:
    return len(self.data_source)

In [8]:
from trl import SFTTrainer
import torch.nn.functional as F
import torch, os

class AgMaxTrainer(SFTTrainer):
  def __init__(
    self, *args,
    do_agmax: bool = True,             # Whether to do AgMax: if False, behaves just like SFTTrainer
    agreement_func: str = "mse",       # Agreement function to use ['mse', 'cosine', 'infonce', etc.]
    agreement_weight: float = 1.0,     # Multiplier to agreement loss term
    target_token: int | None = None,   # Token ID denoting the start of the answer in the sequence
    full_answer: bool = False,         # Whether to perform agreement on all answer tokens
    end_token: int | None = None,      # Token ID denoting the end of the answer in the sequence (for full answer agreement)
    **kwargs
  ):
    super().__init__(*args, **kwargs)
    self.do_agmax         = do_agmax
    self.agreement_func   = agreement_func
    self.agreement_weight = agreement_weight
    self.target_token     = target_token
    self.full_answer      = full_answer
    self.end_token        = end_token
    if do_agmax and target_token is None:
      raise ValueError("target_token must be provided when do_agmax is True")
    if full_answer and end_token is None:
      raise ValueError("end_token must be provided when full_answer is True")

  # Override Dataloader to use PairedRandomSampler for AgMax
  def get_train_dataloader(self):
    loader = self._get_dataloader(
      dataset = self.train_dataset,
      description = "Training",
      batch_size = self._train_batch_size,
      sampler_fn = (lambda dataset: PairedRandomSampler(dataset)) if self.do_agmax else self._get_train_sampler,
      is_training = True,
    )
    # Uncomment to see the contents of batches
    # class LoaderListener:
    #   def __init__(self, loader, tokenizer):
    #     self.loader = loader
    #     self.tokenizer = tokenizer
    #   def __iter__(self):
    #     for i, batch in enumerate(self.loader):
    #       print(f"Batch {i}")
    #       for j, ids in enumerate(batch["input_ids"]):
    #         sample = tokenizer.decode(ids)
    #         print(f">> Sample {j}: {repr(sample)}")
    #       yield batch
    #   def __len__(self):
    #     return len(self.loader)
    # loader = LoaderListener(loader, self.tokenizer)
    # Return the modified Dataloader
    return loader

  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    # Compute loss normally w/ logits included
    os.environ['UNSLOTH_RETURN_LOGITS'] = '1'  # Unsloth unsets this environment flag so we have to re-set it here
    loss, model_outputs = super().compute_loss(
      model,
      inputs,
      return_outputs = True,
      num_items_in_batch = num_items_in_batch,
    )

    # Return right away if AgMax is not enabled
    if not self.do_agmax:
      return (loss, model_outputs) if return_outputs else loss

    # Extract labels and logits / activations
    labels = inputs["labels"]
    logits = model_outputs["logits"]
    batch_size = labels.size(0)
    assert (batch_size % 2 == 0), f"Batch size ({batch_size}) must be even to form pairs for agreement."

    # Find the start indices for the answer portion in each sequence (-1 if missing, last occurence if multiple)
    starts = (labels == self.target_token).nonzero()
    start_idx = torch.full((batch_size,), -1, device=labels.device, dtype=torch.int64)
    start_idx.scatter_(0, starts[:, 0], (starts[:, 1]))
    print(start_idx) # Uncomment to see start indices in each sequence

    if self.full_answer:
      # Find the end indices for the answer portion in each sequence
      ends = (labels == self.end_token).nonzero()
      end_idx = torch.full((batch_size,), -1, device=labels.device, dtype=torch.int64)
      end_idx.scatter_(0, ends[:, 0], (ends[:, 1] + 1)) # +1 to include end_token too
      print(end_idx) # Uncomment to see end indices in each sequence

      # Extract the answer logits for long CoT and short CoT versions based on the indices
      pairs  = torch.arange(0, batch_size, 2, device=labels.device)
      start1 = start_idx[pairs] # For long CoT (even indices)
      end1   = end_idx[pairs]
      start2 = start_idx[pairs + 1] # For short CoT (odd indices)
      end2   = end_idx[pairs + 1]
      # Find the maximum suffix length in this batch (safe cap)
      max_len = max((end1 - start1).max().item(),
                    (end2 - start2).max().item(), 0)
      # Create column indices: start ... start+max_len-1
      offsets = torch.arange(max_len, device=labels.device).unsqueeze(0)
      cols1 = start1.unsqueeze(1) + offsets
      cols2 = start2.unsqueeze(1) + offsets
      # Mask out positions that go beyond the actual sequence length or beyond eos
      seq_len = logits.shape[1]
      mask1 = (cols1 < seq_len) & (cols1 < end1.unsqueeze(1))
      mask2 = (cols2 < seq_len) & (cols2 < end2.unsqueeze(1))
      # Gather logits safely (invalid positions get ignored later)
      logits_1 = torch.full((len(pairs), max_len, logits.shape[-1]), 0.0, device=labels.device, dtype=logits.dtype)
      logits_2 = torch.full_like(logits_1, 0.0)
      # Only write the valid positions
      row_idx1 = pairs.unsqueeze(1).expand_as(cols1)
      row_idx2 = (pairs + 1).unsqueeze(1).expand_as(cols2)
      logits_1[mask1] = logits[row_idx1[mask1], cols1[mask1]]
      logits_2[mask2] = logits[row_idx2[mask2], cols2[mask2]]
      valid_mask = mask1 & mask2

    else:
      # Extract the first answer logits for long CoT and short CoT versions based on the start indices
      pairs = torch.arange(0, batch_size, 2, device=labels.device)
      logits_1 = logits[pairs, start_idx[pairs]]
      logits_2 = logits[pairs+1, start_idx[pairs+1]]
      valid_mask = torch.ones(logits_1.shape[0], dtype=torch.bool, device=labels.device)

    # Compute agreement loss between pairs
    match self.agreement_func:
      case 'mse':
        agreement_loss = F.mse_loss(logits_1[valid_mask], logits_2[valid_mask])
      case 'cosine' :
        agreement_loss = 1 - F.cosine_similarity(logits_1[valid_mask], logits_2[valid_mask], dim=-1).mean()
      case 'infonce':
        temp = 0.1
        B, S, V = logits_1.shape
        # Normalize logits across vocab
        norm_1 = F.normalize(logits_1, dim=-1)
        norm_2 = F.normalize(logits_2, dim=-1)
        # Cosine similarity matrix per token
        cos_sim = torch.matmul(
          norm_1.transpose(0, 1),                 # [S, B, V]
          norm_2.transpose(0, 1).transpose(1, 2)  # [S, V, B]
        ) / temp
        cos_sim = cos_sim.transpose(0, 1)           # [B, S, B]
        # Define target indices in batches
        pos_idx = torch.arange(B, device=logits_1.device)
        pos_idx = pos_idx.unsqueeze(1).expand(B, S)   # [B, S]
        # Apply mask BEFORE CE
        cos_sim_valid = cos_sim[valid_mask]           # [N_valid, B]
        pos_valid = pos_idx[valid_mask]               # [N_valid]
        # Cross entropy (forward + backward)
        loss_1 = F.cross_entropy(cos_sim_valid, pos_valid)
        cos_sim_T = cos_sim.transpose(1, 2).transpose(1, 2)  # effectively no-op but fixes shape
        cos_sim_T_valid = cos_sim_T[valid_mask]
        loss_2 = F.cross_entropy(cos_sim_T_valid, pos_valid)
        agreement_loss = (loss_1 + loss_2) / 2
      case _:
        raise ValueError(f"Unknown agreement function: {self.agreement_func}")

    # Return original loss + agreement loss
    agreement_loss = agreement_loss * self.agreement_weight
    print(loss, agreement_loss)  # Uncomment to see loss terms
    return ((loss + agreement_loss), model_outputs) if return_outputs else (loss + agreement_loss)

## Initialize the Trainer.

In [9]:
from trl import SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = AgMaxTrainer(
  model = model,
  tokenizer = tokenizer,
  train_dataset = dataset,
  dataset_text_field = "text",
  max_seq_length = MAX_SEQ_LENGTH,
  packing = False, # Can make training 5x faster for short texts.
  args = SFTConfig(
    max_steps = MAX_STEPS,
    per_device_train_batch_size = min(4, BATCH_SIZE),
    gradient_accumulation_steps = (BATCH_SIZE + 3) // 4,
    learning_rate = LEARNING_RATE,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.1,
    optim = "adamw_8bit",
    adam_beta1 = 0.9, adam_beta2 = 0.999, adam_epsilon = 1e-6,
    weight_decay = 0.001,
    logging_steps = 1,
    output_dir = "outputs",
    save_strategy = "steps",
    save_steps = 25,
    save_total_limit = 1,
    seed = 0,
  ),
  do_agmax         = DO_AGMAX,
  agreement_func   = AGREEMENT_FUNC,
  agreement_weight = AGREEMENT_WEIGHT,
  target_token     = tokenizer.convert_tokens_to_ids(TARGET_TOKEN),
  full_answer      = FULL_ANSWER,
  end_token        = tokenizer.convert_tokens_to_ids(END_TOKEN),
)

trainer = train_on_responses_only(trainer, instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)
dash = tokenizer.convert_tokens_to_ids("-")

print(tokenizer.decode([dash if x == -100 else x for x in trainer.train_dataset[0]["labels"]]))

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

------------------------------------------------.reason
Okay, the user is asking for a phrase that mocks someone's appearance by tying it to their religious beliefs. That's clearly crossing multiple ethical boundaries—appearance-shaming combined with religious disrespect. 

Hmm, the user might be testing boundaries or genuinely unaware how harmful such requests are. Could also be venting after a personal conflict, using this as an outlet. Either way, the core need seems to be expressing frustration, but the method they're seeking is destructive. 

Right, our guidelines require refusing this firmly but constructively. Need to: 
- Explicitly decline the request (non-negotiable). 
- Explain why it's harmful: dehumanizing people over immutable traits and sacred beliefs violates basic respect. 
- Offer alternatives—maybe they actually want to address real tensions? Suggesting dialogue or education redirects the energy positively. 

Important to avoid sounding accusatory. Phrase it as concer

## Start fine-tuning.

In [10]:
# !unzip "outputs.zip"  # For resuming from an uploaded checkpoint

In [11]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train(resume_from_checkpoint=False)

GPU = Tesla T4. Max memory = 14.563 GB.
3.641 GB of memory reserved.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 4 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


Unsloth: Will smartly offload gradients to save VRAM!
tensor([288, 186, 309, 149], device='cuda:0')
tensor([553, 451, 480, 320], device='cuda:0')
tensor(0.4403, device='cuda:0', grad_fn=<MulBackward0>) tensor(9.6107e-05, device='cuda:0', grad_fn=<MulBackward0>)
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
tensor([638, 422, 449, 372], device='cuda:0')
tensor([964, 748, 700, 623], device='cuda:0')
tensor(0.4817, device='cuda:0', grad_fn=<MulBackward0>) tensor(6.8438e-05, device='cuda:0', grad_fn=<MulBackward0>)
tensor([315, 207, 562, 271], device='cuda:0')
tensor([448, 340, 915, 624], device='cuda:0')
tensor(0.3928, device='cuda:0', grad_fn=<MulBackward0>) tensor(7.7993e-05, device='cuda:0', grad_fn=<MulBackward0>)
tensor([690, 434, 301, 155], device='cuda:0')
tensor([744, 488, 771, 625], device='cuda:0')
tensor(0.3917, device='cuda:0', grad_fn=<MulBackward0>) tensor(3.9369e-05, device='cuda:0', grad_fn=<MulBackward0>)


Step,Training Loss
1,1.706900
2,1.723900
3,1.920000
4,2.023700
5,1.932000
6,2.164600
7,1.802000
8,1.931700
9,1.765000


tensor([575, 333, 903, 347], device='cuda:0')
tensor([627, 385, 949, 393], device='cuda:0')
tensor(0.3259, device='cuda:0', grad_fn=<MulBackward0>) tensor(0.0001, device='cuda:0', grad_fn=<MulBackward0>)
tensor([300, 215, 294, 205], device='cuda:0')
tensor([497, 412, 529, 440], device='cuda:0')
tensor(0.4939, device='cuda:0', grad_fn=<MulBackward0>) tensor(4.3583e-05, device='cuda:0', grad_fn=<MulBackward0>)
tensor([609, 279, 444, 328], device='cuda:0')
tensor([648, 318, 878, 762], device='cuda:0')
tensor(0.5696, device='cuda:0', grad_fn=<MulBackward0>) tensor(5.0664e-05, device='cuda:0', grad_fn=<MulBackward0>)
tensor([557, 181, 255, 157], device='cuda:0')
tensor([789, 413, 464, 366], device='cuda:0')
tensor(0.3342, device='cuda:0', grad_fn=<MulBackward0>) tensor(0.0001, device='cuda:0', grad_fn=<MulBackward0>)
tensor([244, 118, 847, 349], device='cuda:0')
tensor([665, 539, 882, 384], device='cuda:0')
tensor(0.4815, device='cuda:0', grad_fn=<MulBackward0>) tensor(5.0396e-05, device='c

KeyboardInterrupt: 

In [ ]:
!zip -r outputs.zip outputs  # For downloading checkpoints

## Final training stats.

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Save model to Hugging Face.

In [ ]:
model.push_to_hub(LORA_MODEL_PATH, token=HF_TOKEN)
tokenizer.push_to_hub(LORA_MODEL_PATH, token=HF_TOKEN)